## Port Connectivity — `nc` (netcat)

**Support context:** `ping` and `traceroute` prove a host is *reachable*. Neither says whether the **service** on it accepts connections. Your DNS-ticket scenario had this gap: host alive, service down. `nc -zv host port` answers **"can I complete a TCP handshake to this port?"** in one line, and **sorts the failure into your 3-bucket model**.

**Developer bridge:** like `socket.connect((host, port))` in Python, from the shell. `-z` means connect, then close without sending anything.

**COMMAND:** `nc`

**SYNTAX:** `nc [flags] host port` (or `host port-range`)

| Flag | Meaning |
|---|---|
| `-z` | Zero-I/O mode: connect, report, close. This is the port check. |
| `-v` | Verbose. Without it, `-z` prints nothing and you only get the exit code. |
| `-w N` | Timeout in seconds. Without it, a filtered port can hang for a long time. |
| `-l` | Listen mode. Makes you the server, useful for testing from the other side. |
| `-p` | Local port to listen on. Needed on some variants, e.g. `nc -l -p 9000`. |
| `-u` | UDP instead of TCP. UDP "success" is unreliable, see the gotchas. |

**Live demo (no sudo needed)**. Run these now.

**1. Test a port that should be open**:
```bash
nc -zv -w 3 8.8.8.8 53
nc -zv -w 3 google.com 443
```
Expected output:
```bash
dennie@X390:~/linux-tut$ nc -zv -w 3 8.8.8.8 53
Connection to 8.8.8.8 53 port [tcp/domain] succeeded!

dennie@X390:~/linux-tut$ nc -zv -w 3 google.com 443
Connection to google.com (142.251.38.142) 443 port [tcp/https] succeeded!
```
**2. Test a port that should be closed (your own machine)**:
```bash
nc -zv -w 3 localhost 9999
```
Expected output:
```bash
dennie@X390:~/linux-tut$ nc -zv -w 3 localhost 9999
nc: connect to localhost (127.0.0.1) port 9999 (tcp) failed: Connection refused
```
**3. Run both sides yourself**.

Use two terminals.

Terminal A is the "service":
```bash
nc -l 9999
```
```bash
dennie@X390:~/linux-tut$ nc -l 9999
hello
```

Terminal B is the "client":
```bash
nc -zv -w 3 localhost 9999    # now succeeds
nc localhost 9999             # type "hello" + Enter, it appears in Terminal A
```
```bash
dennie@X390:~/linux-tut$ nc localhost 9999 
hello
```
Stop the listener with Ctrl+C in Terminal A, then re-run the check from Terminal B. It's back to `Connection refused`. You've just reproduced "service down" on demand.

**4. Scan a few ports**:
```bash
nc -zv -w 2 localhost 20-25

dennie@X390:~/linux-tut$ nc -zv -w 2 localhost 20-25
nc: connect to localhost (127.0.0.1) port 20 (tcp) failed: Connection refused
nc: connect to localhost (127.0.0.1) port 21 (tcp) failed: Connection refused
nc: connect to localhost (127.0.0.1) port 22 (tcp) failed: Connection refused
nc: connect to localhost (127.0.0.1) port 23 (tcp) failed: Connection refused
nc: connect to localhost (127.0.0.1) port 24 (tcp) failed: Connection refused
nc: connect to localhost (127.0.0.1) port 25 (tcp) failed: Connection refused
```
Expected output is a `failed: Connection refused` line for each closed port.

---

**EXAMPLE 1 — port that should be open:**

In [ ]:
$ nc -zv -w 3 8.8.8.8 53
Connection to 8.8.8.8 53 port [tcp/domain] succeeded!

$ nc -zv -w 3 google.com 443
Connection to google.com 443 port [tcp/https] succeeded!

**EXAMPLE 2 — port that should be closed (your own machine):**

In [ ]:
$ nc -zv -w 3 localhost 9999
nc: connect to localhost port 9999 (tcp) failed: Connection refused

**EXAMPLE 3 — run both sides yourself (two terminals):**

Terminal A is the "service":
```bash
nc -l 9999
```
Terminal B is the "client":
```bash
nc -zv -w 3 localhost 9999    # now succeeds
nc localhost 9999             # type "hello" + Enter, it appears in Terminal A
```
Stop the listener with Ctrl+C in Terminal A, then re-run the check from Terminal B. It's back to `Connection refused`. You've reproduced "service down" on demand.

**EXAMPLE 4 — scan a few ports:**
```bash
nc -zv -w 2 localhost 20-25
```
Expected: a `failed: Connection refused` line for each closed port.

**READ THE OUTPUT:**

| Result | Meaning | Bucket |
|---|---|---|
| `succeeded!` | Handshake completed. Service is listening and the path is open. | n/a |
| `Connection refused` | Host answered with a TCP RST. It's alive, but nothing is listening, or the service is down. **Fast** failure. | Reached but rejected |
| Timeout (hangs until `-w` expires) | Nothing came back. Firewall silently dropping, host down, or broken route. **Slow** failure. | Transport failed |
| `Name or service not known` | DNS failed before any connection was attempted. | Resolution failed |

**Refused vs timeout is the key distinction.**
- **Refused**: host is up, port is closed
- **Timeout**: something is dropping the packets.

**TICKET SCENARIO:** "Our app can't connect to the database server at 10.0.0.5." `ping` replies. `nc -zv -w 3 10.0.0.5 5432` returns `Connection refused`. The host is fine, so PostgreSQL is down or listening elsewhere: move on to `systemctl status postgresql` and `ss -tlnp` on that box. If it had timed out instead, suspect a firewall or security group and never touch the service.

**GOTCHAS**
- Always use `-w`. Without it, a firewalled port makes you wait minutes.
- **Variants differ.** Ubuntu ships `netcat-openbsd`. `-p` with `-l` behaves differently across variants. Check yours with `nc -h 2>&1 | head -3`.
- **UDP:** `nc -zvu` reports "succeeded" when it merely sent a packet and got no ICMP error back. Don't trust it for UDP services like DNS.
- **WSL:** `localhost` in WSL2 is the WSL VM, not Windows. A service on Windows isn't at `localhost` from inside WSL, because it sits behind the virtual NAT. Covered later in the WSL networking section.

```bash
dennie@X390:~/linux-tut$ nc -h 2>&1 | head -3
OpenBSD netcat (Debian patchlevel 1.226-1ubuntu2)
usage: nc [-46CDdFhklNnrStUuvZz] [-I length] [-i interval] [-M ttl]
          [-m minttl] [-O length] [-P proxy_username] [-p source_port]
```
OpenBSD netcat. That's the variant where `nc -l 9999` works without `-p`. The flags in the lesson are correct for your install.

### TRY NOW

1. Run `nc -zv -w 3 google.com 80 443 8080`. Which ports succeed, and is the 8080 result refused or a timeout? What does that tell you about Google's firewall?
2. Run `nc -zv -w 3 10.255.255.254 53`. That's your WSL resolver. Does it answer on TCP 53?

In [ ]:
dennie@X390:~/linux-tut$ nc -zv -w 3 google.com 80 443 8080
Connection to google.com (142.251.141.174) 80 port [tcp/http] succeeded!
Connection to google.com (172.217.19.14) 443 port [tcp/https] succeeded!
nc: connect to google.com (172.217.19.14) port 8080 (tcp) timed out: Operation now in progress
nc: connect to google.com (2a00:1450:4014:801::200e) port 8080 (tcp) failed: Network is unreachable

There are two lines because `google.com` resolves to both an IPv4 and an IPv6 address, and nc tries each.

**Line 1, IPv4 (`172.217.19.14`): timed out.** This is the failure that matters.

- It is a timeout, not `Connection refused`.
- The `real 3.1s` line is the timeout hitting your `-w 3` limit.
- Nothing came back, so a firewall is silently dropping the packets.
- Google's servers only expose the ports they serve (80 and 443). Anything else is dropped rather than refused.
- If 8080 were simply closed on a reachable host, you'd get an instant `Connection refused`.
- A timeout means the packets never reached a listener, and the sender never learned why.

**Line 2, IPv6: `Network is unreachable`.** This is the IPv6-route-gap you hit in the DNS notebook. WSL has no IPv6 route out. The error is instant and local, so it's unrelated to Google's firewall. It would be the same for any IPv6 port, including 443. Use `nc -4` to ignore it.

> **Support rule:** refused is fast and the host answered. A timeout is slow and something is dropping packets. Check the `real` time as well as the message.

In [ ]:
dennie@X390:~/linux-tut$ nc -zv -w 3 10.255.255.254 53
Connection to 10.255.255.254 53 port [tcp/domain] succeeded!

In [ ]:
dennie@X390:~/linux-tut$ time nc -4zv -w 3 google.com 8080
nc: connect to google.com (142.251.38.142) port 8080 (tcp) timed out: Operation now in progress

real    0m3.110s``
user    0m0.008s
sys     0m0.011s

**`time nc -4zv -w 3 google.com 8080`** timed out with no IPv6 line.

- `-4` removed the `Network is unreachable` noise, leaving the single timeout.
- `real 3.110s` is your `-w 3` limit expiring, which confirms nothing came back.
- That's a firewall drop, not a refusal.
- There's a stray `` ` `` after `real 0m3.110s` in your notebook cell. It's harmless, but you may want to delete it.

In [ ]:
dennie@X390:~/linux-tut$ nc -zv -w 3 10.255.255.254 53
Connection to 10.255.255.254 53 port [tcp/domain] succeeded!

**`nc -zv -w 3 10.255.255.254 53`** succeeded.

- WSL's resolver answers on TCP 53 as well as UDP.
- This is the same resolver that choked on the `dig +trace` root priming query in your DNS notebook.
- That failure was in how the resolver handles that query, not a closed port. You can reach it, and it was the query that failed.
- Use `nc` to rule out the port, then `dig` to test actual resolution.

Of the three failure shapes, you've now seen refused and timeout. `Name or service not known` is the third. Try it with a made-up name:

In [ ]:
dennie@X390:~/linux-tut$ nc -zv -w 3 nosuchhost.invalid 443
nc: getaddrinfo for host "nosuchhost.invalid" port 443: Name or service not known

### DRILL

**Q1.** `nc -zv -w 3 10.0.0.5 5432` fails instantly with `Connection refused`. Is the host up? Where do you look next?

**Q2.** The same command hangs for the full 3 seconds, then times out. What do you suspect, and what do you *not* bother checking yet?

**Q3.** Why is `nc -zvu` against a DNS server not proof the service works?

### DRILL ANSWERS

**Q1 — Reached but rejected.** The host is up: it answered with a TCP RST, so nothing is listening on that port or the service is down. Next, on the target: `systemctl status postgresql` (running? enabled on boot?), `ss -tlnp` and `journalctl -u postgresql` (why is it down?).
- 💡 `ss -tlnp` also shows the **bind address**. A service on `127.0.0.1:5432` refuses connections from other hosts but works locally — classic "refused remotely, fine locally".

**Q2 — Transport failed.** Nothing came back, so something is dropping packets: a firewall silently dropping, the host down, or a broken route. **Don't check the service on the target yet** — packets may never reach it, so `systemctl status` could show a perfectly healthy service. Narrow down first:
- `ping` replies: host is up, suspect a firewall.
- `ping` silent: host down or route broken (but remember the gateway that ignores ICMP yet forwards fine).
- `traceroute -n`: shows where the path dies.

**Q3 — UDP has no handshake.** `nc -zvu` only means "I sent a packet and no ICMP error came back". Silence = open **or** filtered; `nc` can't tell which. Test the real protocol instead: `dig @10.255.255.254 google.com`.

🔁 **Support rule:** `nc` proves TCP ports. For UDP, test the actual protocol. Refused is fast (host answered); a timeout is slow (something drops packets) — check the `real` time as well as the message.

**Score: 3/3.**